## PREDICT PRODUCT PRICES


In [ ]:
import os
import re
import math
from tqdm import tqdm
from huggingface_hub import login
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments, Trainer,set_seed
from peft import LoraConfig,PeftModel
from datetime import datetime
from datasets import load_dataset,Dataset,DatasetDict
import matplotlib.pyplot as plt


c:\Users\KRISH\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [16]:
# Tokenizers
LLAMA_3_1="meta-llama/Meta-Llama-3.1-8B"
QWEN_2_5="Qwen/Qwen2.5-7B"
GEMMA_2= "google/gemma-2-9b"
PHI_3="microsoft/Phi-3-medium-4k-instruct"

# Constants
BASE_MODEL= LLAMA_3_1
HF_USER="ed-donner"
DATASET_NAME=F"{HF_USER}/pricer-data"
MAX_SEQUENCE_LENGTH=192
QUANT_4_BIT=True

#  Used for writing to output in colour

GREEN= "\033[92m"
YELLOW="\033[93m"
RED="\033[91m"
RESET="\033[0m"
COLOR_MAP={"red":RED,"green":GREEN,"yellow":YELLOW}




In [7]:
hf_token=os.getenv("HF_TOKEN")

In [8]:
def investigate_tokenizer(model_name):
    print("Investigating tokenizer for model:",model_name)
    tokenizer=AutoTokenizer.from_pretrained(model_name,trust_remote_code=True
                                            )
    for number in [0,1,10,100,999,1000]:
        tokens=tokenizer.encode(str(number),add_special_tokens=False)
        print(f"The tokens for {number}:{tokens}")

    

In [13]:
investigate_tokenizer(PHI_3)

Investigating tokenizer for model: microsoft/Phi-3-medium-4k-instruct
The tokens for 0:[29871, 29900]
The tokens for 1:[29871, 29896]
The tokens for 10:[29871, 29896, 29900]
The tokens for 100:[29871, 29896, 29900, 29900]
The tokens for 999:[29871, 29929, 29929, 29929]
The tokens for 1000:[29871, 29896, 29900, 29900, 29900]


In [17]:
dataset=load_dataset(DATASET_NAME)
train=dataset["train"]
test=dataset["test"]

Generating test split: 100%|██████████| 2000/2000 [00:00<00:00, 202403.38 examples/s]


In [19]:
test[0]

{'text': "How much does this cost to the nearest dollar?\n\nOEM AC Compressor w/A/C Repair Kit For Ford F150 F-150 V8 & Lincoln Mark LT 2007 2008 - BuyAutoParts NEW\nAs one of the world's largest automotive parts suppliers, our parts are trusted every day by mechanics and vehicle owners worldwide. This A/C Compressor and Components Kit is manufactured and tested to the strictest OE standards for unparalleled performance. Built for trouble-free ownership and 100% visually inspected and quality tested, this A/C Compressor and Components Kit is backed by our 100% satisfaction guarantee. Guaranteed Exact Fit for easy installation 100% BRAND NEW, premium ISO/TS 16949 quality - tested to meet or exceed OEM specifications Engineered for superior durability, backed by industry-leading unlimited-mileage warranty Included in this K\n\nPrice is $",
 'price': 374.41}

### Prepare our Base Llama Model for evaluation

In [23]:
if QUANT_4_BIT:
    quant_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4"
    )

else:
    quant_config=BitsAndBytesConfig(
        load_in_8bit=True,
        bnb_8bit_compute_dtype=torch.bfloat16
    )

In [ ]:
tokenizer=AutoTokenizer.from_pretrained(BASE_MODEL,trust_remote_code=True)
tokenizer.pad_token=tokenizer.eos_token
tokenizer.padding_side="right"

base_model=AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)
base_model.generation_config.pad_token_id=tokenizer.pad_token_id
print(f"Memory footprint of the base model: {base_model.get_memory_footprint()/1e9:.1f} GB")

In [25]:
def extract_price(s):
    if "Price is $" in s:
        contents=s.split("Price is $")[1]
        contents=contents.replace(',','').replace('$','')
        match=re.search(r"[-+]?\d*\.\d+|\d+",contents)
        return float(match.group()) if match else 0
    return 0

In [27]:
extract_price("The Price is $9994 tsoccheap man")

9994.0

In [ ]:
def model_predict(prompt):
    set_seed(42)
    inputs= tokenizer.encode(prompt,return_tensors="pt").to("cuda")
    attention_mask=torch.ones(inputs.shape,device="cuda")
    outputs=base_model.generate(inputs,max_new_tokens=4, attention_mask=attention_mask,num_return_sequences=1)
    response=tokenizer.decode(outputs[0])
    return extract_price(response)

### Evaluation

In [ ]:
import math
import matplotlib.pyplot as plt

# Dummy/Fallback styling in case COLOR_MAP isn't defined globally
COLOR_MAP = {"green": "\033[92m", "orange": "\033[93m", "red": "\033[91m"}
RESET = "\033[0m"

class Tester:
    def __init__(self, predictor, data, title="Model Evaluation", size=250):
        self.predictor = predictor
        # Prevent index errors if dataset is smaller than requested size
        self.data =  
        self.size = min(size, len(data))
        self.title = title
        self.guesses = []
        self.truths = []
        self.errors = []
        self.sles = []
        self.colors = []

    def color_for(self, error, truth):
        # Prevent division by zero if truth is 0
        if truth == 0:
            return "green" if error == 0 else "red"
            
        pct_error = error / truth
        
        # Switched to 'and' logic so small cheap items aren't falsely marked green
        if error < 40 and pct_error < 0.2:
            return "green"
        elif error < 80 and pct_error < 0.5:
            return "orange"
        else:
            return "red"
        
    def run_datapoint(self, i):
        datapoint = self.data[i]
        
        # Fallback to "prompt" if "text" key doesn't exist
        text_content = datapoint.get("text", datapoint.get("prompt", ""))
        
        guess = self.predictor(datapoint["prompt"])
        truth = datapoint["price"]
        
        error = abs(guess - truth)
        
        # Math safety: handle potential log(0) issues by adding 1
        log_error = math.log(truth + 1) - math.log(guess + 1)
        sle = log_error ** 2
        
        color = self.color_for(error, truth)
        
        # Safely parse title snippet
        split_text = text_content.split("\n\n")
        raw_title = split_text[1] if len(split_text) > 1 else text_content
        title_snippet = raw_title[:20] + "..."
        
        self.guesses.append(guess)
        self.truths.append(truth)
        self.errors.append(error)
        self.sles.append(sle)
        self.colors.append(color)
        
        print(f"{COLOR_MAP[color]}{title_snippet:20} | Guess: {guess:8.2f} | Truth: {truth:8.2f} | Error: {error:8.2f} | SLE: {sle:8.4f}{RESET}")
    
    def chart(self, title):
        if not self.guesses:
            print("No data to chart.")
            return
            
        plt.figure(figsize=(10, 6))
        max_value = max(max(self.guesses), max(self.truths)) if self.guesses else 10
        
        # Perfect prediction identity line
        plt.plot([0, max_value], [0, max_value], color='deepskyblue', lw=2, alpha=0.6, label='Perfect Matches')
        plt.scatter(self.truths, self.guesses, c=self.colors, alpha=0.6)
        
        plt.xlabel('Ground Truth')
        plt.ylabel('Predicted')
        plt.xlim(0, max_value)
        plt.ylim(0, max_value)
        plt.title(title)
        plt.grid(True, alpha=0.3)
        plt.show()
    
    def report(self):
        if not self.errors:
            print("No data processed to generate a report.")
            return
        average_error = sum(self.errors) / len(self.errors)
        rmsle = math.sqrt(sum(self.sles) / len(self.sles))
        hits = sum(1 for color in self.colors if color == "green")
        
        full_title = f"{self.title}\nAvg Error: {average_error:.2f}, RMSLE: {rmsle:.4f}, Hits: {hits}/{len(self.colors)}"
        self.chart(full_title)
    
    def run(self):
        for i in range(self.size):
            self.run_datapoint(i)
        self.report()
    
    @classmethod
    def test(cls, function, data, title="Model Evaluation", size=250):
        cls(function, data, title=title, size=size).run()

In [ ]:
Tester.test(model_predict,test)